# Posterior Sampling：GPの不確実性から候補提案まで

事後分布からのサンプリングが**どのような不確実性を表すか**を学び、`SobolQMCNormalSampler`・`IIDNormalSampler`・`qLogExpectedImprovement`・Thompson Samplingを比較します。

2点の同時提案から観測・再学習・次の候補提案まで実行します。サンプラーと獲得関数はBoTorch / robotorchanの既存公開APIを利用します。


## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.acquisition.objective import LinearMCObjective
from botorch.optim import optimize_acqf
from botorch.sampling.normal import IIDNormalSampler, SobolQMCNormalSampler

from robotorchan.models import ModelListGP, SingleTaskGP

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 事後分布とサンプラーの役割

GPは、未観測の点に対して**1つの予測値だけでなく確率分布**を返します。`model.posterior(X)` は予測分布を表し、サンプラーはその分布から複数の実現値を生成します。

サンプルは新しい実験データではありません。モデルが現時点であり得ると考える関数値です。実際の観測を増やすには、候補を実験で評価する必要があります。

## 2. Single-output Gaussian posterior

まず最も基本的な`SingleTaskGP`です。

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.linspace(0.05, 0.95, 9).unsqueeze(-1)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()

candidate_X = torch.tensor([[0.25], [0.65]])
posterior = model.posterior(candidate_X)
print("posterior mean shape:", posterior.mean.shape)
print("posterior variance shape:", posterior.variance.shape)

## 3. Sobol QMC and IID normal sampling

Gaussian posteriorならBoTorch native normal samplersを直接利用できます。`sample_shape=[64]`、`q=2`、single-outputなのでsample shapeは`[64, 2, 1]`です。

In [ ]:
qmc_sampler = SobolQMCNormalSampler(torch.Size([64]), seed=123)
iid_sampler = IIDNormalSampler(torch.Size([64]), seed=123)

qmc_samples = qmc_sampler(posterior)
iid_samples = iid_sampler(posterior)

assert qmc_samples.shape == torch.Size([64, 2, 1])
assert iid_samples.shape == torch.Size([64, 2, 1])
print("QMC samples:", qmc_samples.shape)
print("IID samples:", iid_samples.shape)

### QMCとIIDの違い

`IIDNormalSampler` は独立な疑似乱数に基づくMCサンプルを生成します。`SobolQMCNormalSampler` は低食い違いのSobol系列を使い、同じサンプル数でも期待値推定のばらつきを小さくできる場合があります。

両者とも**同じ事後分布を近似**します。QMCを使うこと自体でGPの予測分布や観測データが変わるわけではありません。

## 4. q points are sampled jointly

`q=2`は2点を独立な1点posteriorとして別々にsamplingする意味ではありません。posterior covarianceを保持したままjoint sampleを生成します。

In [ ]:
sample_covariance = torch.cov(qmc_samples[..., 0].transpose(0, 1))
print("empirical covariance across q points:")
print(sample_covariance)

### 2候補のサンプルは相関を持つ

`q=2` では、2点の予測を別々の分布から独立に抽出するのではなく、**2点を含む同時事後分布**からサンプリングします。近い入力ほど予測誤差が相関することがあり、獲得関数が2候補を同時評価する際に重要です。

次の経験的共分散は有限個のサンプルからの推定値です。モデルが保持する理論上の共分散と完全一致するわけではありません。

In [ ]:
with torch.no_grad():
    exact_covariance = posterior.mvn.covariance_matrix
print("GP posterior covariance across 2 points:")
print(exact_covariance)
print("QMC sample covariance across 2 points:")
print(torch.cov(qmc_samples[..., 0].transpose(0, 1)))
assert exact_covariance.shape == (2, 2)

## 5. Multi-output ModelListGP is still a joint Gaussian path

`ModelListGP`という名前だけを見て`ListSampler`を選びません。homogeneous Gaussian submodelsではjoint multi-output Gaussian posteriorが返り、normal samplerを使えます。

In [ ]:
second_Y = torch.cos(2 * torch.pi * train_X)
multi_model = ModelListGP(
    SingleTaskGP(train_X, train_Y),
    SingleTaskGP(train_X, second_Y),
)
multi_model.fit()
multi_model.eval()

multi_posterior = multi_model.posterior(candidate_X)
multi_samples = SobolQMCNormalSampler(
    torch.Size([32]),
    seed=456,
)(multi_posterior)

assert multi_samples.shape == torch.Size([32, 2, 2])
print("multi-output posterior mean:", multi_posterior.mean.shape)
print("joint samples:", multi_samples.shape)

### 複数出力と複数候補は別の軸

`ModelListGP` はここでは2つの独立したGPを組み合わせた2出力モデルです。サンプルの `[32, 2, 2]` は、順に **32サンプル × 2候補 × 2出力**を表します。

「2出力だから候補も2点」という意味ではありません。また `ModelListGP` というモデル名だけで `ListSampler` を選ぶ必要もありません。

## 6. Sampling does not define optimization utility

samplerはposterior uncertaintyをsampleするだけです。multi-output samplesをどうdecision utilityへ変換するかはObjectiveの責務です。

### Objectiveはサンプルを評価指標に変換する

`LinearMCObjective(weights=[0.7,0.3])` は、各サンプルの2出力を重み付き和に変換します。サンプラーは**予測の不確実性**を表し、Objectiveは**何を良い結果とみなすか**を決めます。

線形・非線形のObjectiveの詳細はNotebook 36で扱います。

In [ ]:
weights = torch.tensor([0.7, 0.3])
objective = LinearMCObjective(weights=weights)
utility_samples = objective(multi_samples)

assert utility_samples.shape == torch.Size([32, 2])
print("raw multi-output samples:", multi_samples.shape)
print("scalar utility samples:", utility_samples.shape)

## 7. Sampler → Objective → MC acquisition → candidate

sampling contractをそのままqLogEIへ接続し、実際にcandidateを生成します。

### MC獲得関数で2点を同時提案する

`qLogExpectedImprovement` は事後サンプルから、2候補を同時に評価したときの改善を推定します。`best_f` は**観測済み2出力の重み付き効用の最大値**です。

`optimize_acqf(q=2)` が返す2点は、各点を単独で評価した上位2点とは限りません。

In [ ]:
train_multi_Y = torch.cat([train_Y, second_Y], dim=-1)
best_f = (train_multi_Y @ weights).max()
acquisition = qLogExpectedImprovement(
    model=multi_model,
    best_f=best_f,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=789),
    objective=objective,
)
bounds = torch.tensor([[0.0], [1.0]])
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert candidate.shape == torch.Size([2, 1])
print("joint q=2 candidate:", candidate.squeeze(-1))
print("qLogEI value:", value)

### qLogEIで選んだ2候補を実験で評価する

`candidate` は2行の入力です。各候補について2つの生の応答を測定し、観測データに追加します。サンプルから得た仮想的な値を訓練データへ入れるわけではありません。

In [ ]:
def second_response(X: torch.Tensor) -> torch.Tensor:
    return torch.cos(2 * torch.pi * X)

new_Y = torch.cat([response(candidate), second_response(candidate)], dim=-1)
updated_X = torch.cat([train_X, candidate], dim=0)
updated_multi_Y = torch.cat([train_multi_Y, new_Y], dim=0)
updated_model = ModelListGP(
    SingleTaskGP(updated_X, updated_multi_Y[:, [0]]),
    SingleTaskGP(updated_X, updated_multi_Y[:, [1]]),
)
updated_model.fit()
updated_model.eval()
print("New measured outputs:", new_Y)
print("Training points before/after:", train_X.shape[0], updated_X.shape[0])
assert updated_X.shape[0] == train_X.shape[0] + 2

### 再学習後の事後分布と次の提案

新しい観測によってGPの事後分布が更新されます。更新後のモデルを使って**新しいqLogEI**を構成し、次の2点を提案します。

In [ ]:
updated_best_f = (updated_multi_Y @ weights).max()
next_acquisition = qLogExpectedImprovement(
    model=updated_model,
    best_f=updated_best_f,
    sampler=SobolQMCNormalSampler(torch.Size([128]), seed=790),
    objective=objective,
)
next_candidates, next_value = optimize_acqf(
    acq_function=next_acquisition,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)
print("Next joint q=2 candidates:", next_candidates.squeeze(-1))
print("Next qLogEI:", next_value)
assert next_candidates.shape == (2, 1)

### 事後平均と不確実性の変化を可視化する

2つの出力の予測平均と標準偏差を、観測前後で比較します。観測した点の近くで予測が変わる様子を確認してください。

In [ ]:
import matplotlib.pyplot as plt

plot_X = torch.linspace(0.0, 1.0, 201).unsqueeze(-1)
with torch.no_grad():
    before_posterior = multi_model.posterior(plot_X)
    after_posterior = updated_model.posterior(plot_X)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for k, ax in enumerate(axes):
    for post, label in ((before_posterior, "Before"), (after_posterior, "After")):
        center = post.mean[:, k]
        std = post.variance[:, k].clamp_min(0).sqrt()
        ax.plot(plot_X.squeeze(-1), center, label=f"{label} mean")
        ax.fill_between(plot_X.squeeze(-1), center - 2 * std, center + 2 * std, alpha=0.12)
    ax.scatter(candidate.squeeze(-1), new_Y[:, k], marker="x", s=70, label="New observations")
    ax.set(xlabel="x", title=f"Output {k + 1}")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. Candidate gradients are a separate capability

samplerがposteriorをsampleできることと、candidate `X`に対してgradient-based acquisition optimizationできることは別contractです。Gaussian GP pathでは現在のtested contractとしてgradientが保存されます。

In [ ]:
gradient_X = torch.tensor([[[0.42]]], requires_grad=True)
gradient_value = acquisition(gradient_X)
gradient = torch.autograd.grad(gradient_value.sum(), gradient_X)[0]

assert torch.isfinite(gradient).all()
print("candidate gradient:", gradient)

## 9. Thompson sampling is candidate generation, not MC integration

finite poolからposterior drawに基づいてcandidateを選ぶThompson samplingは、acquisition expectationを近似するMC samplerとは別の役割です。robotorchanでは`select_thompson_candidates`がBoTorch `MaxPosteriorSampling`へ委譲します。

### Thompson SamplingはMC獲得関数とは異なる

Thompson Samplingは事後分布から**関数の実現を引いて、その関数上で良い候補を選ぶ**方法です。ここでは101点の有限候補集合から3点を選びます。

一方、`qLogEI` のサンプラーは、獲得関数の期待値を数値的に計算するために使います。同じposterior samplingでも役割が異なります。

In [ ]:
from robotorchan.acquisition import select_thompson_candidates

choices = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
selected = select_thompson_candidates(
    model=model,
    choices=choices,
    num_samples=3,
    replacement=False,
)

assert selected.shape == torch.Size([3, 1])
print("Thompson-selected candidates:", selected.squeeze(-1))

## 10. Posterior-to-sampler map

| Posterior semantics | Preferred sampler | Typical path |
| --- | --- | --- |
| Gaussian | `SobolQMCNormalSampler` | GP, MultiTask, Kronecker, mixed, MF |
| Gaussian | `IIDNormalSampler` | ordinary IID MC when desired |
| empirical ensemble | `IndexSampler` | RF / ExtraTrees style posterior |
| stochastic trajectory | `StochasticSampler` | DeepGP family |
| actual `PosteriorList` | `ListSampler` | only when posterior really is a list |

`make_model_sampler`はcapability metadataに基づくintegration helperですが、direct BoTorch sampler利用を置き換えるpublic abstractionではありません。

## 11. Sampling axes and uncertainty axes

```text
candidate X
  ↓
input perturbation scenarios       ← candidate uncertainty layer
  ↓
posterior
  ↓
MC posterior samples               ← model uncertainty layer
  ↓
objective
  ↓
risk aggregation over scenarios    ← decision-risk layer
```

scenario axisとposterior MC sample axisを同じものとして扱いません。

## 12. Important boundaries

- model family名ではなくposterior semanticsでsamplerを選びます。
- `ModelListGP`だから`ListSampler`とは限りません。
- multi-output / MultiTask / Kronecker posteriorは利用可能なjoint covarianceを分解しません。
- sampling compatibilityはgradient optimization compatibilityを保証しません。
- tree ensembleや外部estimatorはsample可能でもNumPy境界によりcandidate gradientを持たない場合があります。
- Thompson samplingはfinite-pool candidate generationで、MC acquisition integrationとは別です。
- input perturbation scenario生成はposterior samplerの責務ではありません。
- 現在の監査済みmodel setにはrobotorchan独自samplerを追加する必要はありません。

## 13. Related documentation

- `docs/optimization/posterior-sampling.md`
- `docs/optimization/objective-posterior-transform.md`
- `36_objective_posterior_transform.ipynb`

Sampling layerの役割はposterior uncertaintyを正しいjoint semanticsでdrawすることです。candidateの価値やriskをsampling layer自身に持ち込みません。

## Thompson候補の目的関数評価

有限候補集合から選択した3点について、実際の目的関数値を計算します。これはposterior samplingと観測評価を接続する例です。


In [ ]:
thompson_Y = response(selected)
assert thompson_Y.shape[0] == selected.shape[0]
print("Thompson candidate evaluations:", thompson_Y)


## 14. 既存APIの対応表

| 目的 | API | 役割 |
|---|---|---|
| 事後分布を取得 | `model.posterior(X)` | 候補点の予測分布 |
| QMCサンプリング | `SobolQMCNormalSampler` | 低食い違い系列によるMC |
| IIDサンプリング | `IIDNormalSampler` | 疑似乱数によるMC |
| 複数出力の評価 | `LinearMCObjective` | サンプルをスカラー効用へ変換 |
| 2候補の同時提案 | `qLogExpectedImprovement` + `optimize_acqf` | joint q-batch改善 |
| Thompson候補選択 | `select_thompson_candidates` | 事後サンプルに基づく有限候補選択 |
| 観測後の再学習 | `ModelListGP.fit()` | 新しい生の応答でGPを更新 |

**注意：** 事後サンプルは観測データではありません。サンプラーの選択、評価指標の設定、候補最適化、実際の観測は、それぞれ別の役割を持ちます。